# 01.8 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/01-working-with-data/08-statistics-by-hand/exercises.ipynb)

## Exercise 1: by hand first

Values: 5, 7, 8, 12, 14, 15, 40. Compute mean, median, standard deviation, Q1, Q3, IQR. Which value does Tukey's rule flag? Then confirm in code.

In [ ]:
# your code here

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

raw = pd.read_excel(ensure_online_retail(), sheet_name="Year 2010-2011")
print(raw.shape)
raw.head()

In [ ]:
def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df.copy()
    out = out[~out["Invoice"].astype(str).str.startswith("C")]   # drop cancellations
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)]         # drop returns / free items
    out = out.dropna(subset=["Customer ID"])                       # drop guest checkouts
    out = out.drop_duplicates()
    out["Customer ID"] = out["Customer ID"].astype(int)
    out["Revenue"] = out["Quantity"] * out["Price"]
    return out

sales = clean_retail(raw)
print(f"{len(raw):,} rows → {len(sales):,} after cleaning")

In [ ]:
order_values = sales.groupby("Invoice")["Revenue"].sum()
customers = sales.groupby("Customer ID").agg(
    orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
    products=("StockCode", "nunique"), items=("Quantity", "sum"))
customers["avg_order"] = customers["spend"] / customers["orders"]

## Exercise 2: mean vs median per country

Per-customer `spend`: mean and median for the six biggest countries by revenue. Where is the gap largest and why?

In [ ]:
# your code here

## Exercise 3: correlation with and without giants

$r$(`orders`, `spend`) for all customers, then excluding the top 1% by spend. Then try `products` vs `spend`.

In [ ]:
# your code here

## Exercise 4: your own std()

Write `my_std(values)` using the $n-1$ formula and compare with `Series.std()` on 1,000 random order values (`order_values.sample(1000, random_state=0)`).

In [ ]:
# your code here